In [ ]:
import pandas as pd

regression_df = pd.read_csv(
    r"C:\Users\palla\OneDrive\Documents\Coding Projects\Hospital Diabetes Dataset\hospital-readmissions-sql-eda\outputs\05_model_dataset.csv"
)

print(regression_df.head())
print(regression_df.info())
print(regression_df.describe())


   readmit_binary      age dx_bucket med_bucket
0               0   [0-10)    Low Dx   Low Meds
1               0  [10-20)   High Dx   Med Meds
2               0  [20-30)    Med Dx   Med Meds
3               0  [30-40)   High Dx   Med Meds
4               0  [40-50)    Med Dx   Low Meds
<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   readmit_binary  101766 non-null  int64
 1   age             101766 non-null  str  
 2   dx_bucket       101766 non-null  str  
 3   med_bucket      101766 non-null  str  
dtypes: int64(1), str(3)
memory usage: 3.1 MB
None
       readmit_binary
count   101766.000000
mean         0.111599
std          0.314874
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000


In [10]:
print(regression_df.shape)

print("\nTarget distribution:")
print(regression_df["readmit_binary"].value_counts())
print("\nTarget proportion:")
print(regression_df["readmit_binary"].value_counts(normalize=True))


(101766, 4)

Target distribution:
readmit_binary
0    90409
1    11357
Name: count, dtype: int64

Target proportion:
readmit_binary
0    0.888401
1    0.111599
Name: proportion, dtype: float64


In [11]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    average_precision_score
)

# -----------------------------
# 1) Define features + target
# -----------------------------
X_features = regression_df.drop(columns=["readmit_binary"])
y_target = regression_df["readmit_binary"].astype(int)

categorical_columns = X_features.columns.tolist()

# -----------------------------
# 2) Preprocess + model pipeline
# -----------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_columns)
    ],
    remainder="drop"
)

logistic_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

model_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", logistic_model)
])

# -----------------------------
# 3) Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_target,
    test_size=0.2,
    random_state=42,
    stratify=y_target
)

# -----------------------------
# 4) Fit
# -----------------------------
model_pipeline.fit(X_train, y_train)

# -----------------------------
# 5) Predict probabilities
# -----------------------------
probabilities = model_pipeline.predict_proba(X_test)[:, 1]

# -----------------------------
# 6) Core metrics (threshold-free)
# -----------------------------
roc_auc = roc_auc_score(y_test, probabilities)
pr_auc = average_precision_score(y_test, probabilities)

print("ROC AUC:", round(roc_auc, 4))
print("PR AUC (Average Precision):", round(pr_auc, 4))

# -----------------------------
# 7) Threshold option A: 0.50
# -----------------------------
threshold_50 = 0.50
pred_50 = (probabilities >= threshold_50).astype(int)

print("\n--- Threshold = 0.50 ---")
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_50))
print("\nClassification Report:\n", classification_report(y_test, pred_50))

# -----------------------------
# 8) Threshold option B: top 10% risk (capacity-based)
# -----------------------------
top_pct = 0.10
cutoff_index = int(len(probabilities) * (1 - top_pct))  # 90th percentile cutoff
threshold_top10 = np.sort(probabilities)[cutoff_index]

pred_top10 = (probabilities >= threshold_top10).astype(int)

print("\n--- Threshold = Top 10% Risk ---")
print("Top 10% threshold value:", round(float(threshold_top10), 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_top10))
print("\nClassification Report:\n", classification_report(y_test, pred_top10))

# Extra: what is the actual readmission rate among top 10%?
risk_df = pd.DataFrame({"actual": y_test.values, "proba": probabilities})
top10 = risk_df.sort_values("proba", ascending=False).head(int(len(risk_df) * top_pct))
print("\nReadmission rate in top 10% highest-risk group:", round(top10["actual"].mean(), 4))
print("Overall readmission rate (test set):", round(y_test.mean(), 4))


ROC AUC: 0.5647
PR AUC (Average Precision): 0.1312

--- Threshold = 0.50 ---
Confusion Matrix:
 [[8727 9356]
 [ 852 1419]]

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.48      0.63     18083
           1       0.13      0.62      0.22      2271

    accuracy                           0.50     20354
   macro avg       0.52      0.55      0.42     20354
weighted avg       0.82      0.50      0.58     20354


--- Threshold = Top 10% Risk ---
Top 10% threshold value: 0.5459
Confusion Matrix:
 [[16190  1893]
 [ 1955   316]]

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.90      0.89     18083
           1       0.14      0.14      0.14      2271

    accuracy                           0.81     20354
   macro avg       0.52      0.52      0.52     20354
weighted avg       0.81      0.81      0.81     20354


Readmission rate in top 10% highest-risk group: 0.1415
O

In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# 1) Load
full_regression_df = pd.read_csv(
    r"C:\Users\palla\OneDrive\Documents\Coding Projects\Hospital Diabetes Dataset\hospital-readmissions-sql-eda\outputs\06_model_dataset_full.csv"
)

# 2) Basic cleanup: treat blank strings as missing
full_regression_df = full_regression_df.replace(r"^\s*$", np.nan, regex=True)

# 3) Split X/y
X_features = full_regression_df.drop(columns=["readmit_binary"])
y_target = full_regression_df["readmit_binary"].astype(int)

# 4) Define numeric vs categorical columns
numeric_cols = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

categorical_cols = [c for c in X_features.columns if c not in numeric_cols]

# 5) Preprocess + model
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("scaler", StandardScaler())
        ]), numeric_cols),
        ("cat", Pipeline(steps=[
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols),
    ]
)

logistic_model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced"
)

model_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", logistic_model)
])

# 6) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_target,
    test_size=0.2,
    random_state=42,
    stratify=y_target
)

# 7) Fit
model_pipeline.fit(X_train, y_train)

# 8) Probabilities
probabilities = model_pipeline.predict_proba(X_test)[:, 1]

# 9) Threshold-free metrics
roc_auc = roc_auc_score(y_test, probabilities)
pr_auc = average_precision_score(y_test, probabilities)

print("ROC AUC:", round(roc_auc, 4))
print("PR AUC (Average Precision):", round(pr_auc, 4))

# ----------------------------
# Threshold A: 0.50 (reference)
# ----------------------------
threshold_50 = 0.50
pred_50 = (probabilities >= threshold_50).astype(int)

print("\n--- Threshold = 0.50 ---")
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_50))
print("\nClassification Report:\n", classification_report(y_test, pred_50))

# --------------------------------------
# Threshold B: Top 10% risk (operational)
# --------------------------------------
top_pct = 0.10
cutoff_index = int(len(probabilities) * (1 - top_pct))  # 90th percentile
threshold_top10 = np.sort(probabilities)[cutoff_index]
pred_top10 = (probabilities >= threshold_top10).astype(int)

print("\n--- Threshold = Top 10% Risk ---")
print("Top 10% threshold value:", round(float(threshold_top10), 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_top10))
print("\nClassification Report:\n", classification_report(y_test, pred_top10))

risk_df = pd.DataFrame({"actual": y_test.values, "proba": probabilities})
top10 = risk_df.sort_values("proba", ascending=False).head(int(len(risk_df) * top_pct))

print("\nReadmission rate in top 10% highest-risk group:", round(top10["actual"].mean(), 4))
print("Overall readmission rate (test set):", round(y_test.mean(), 4))


ROC AUC: 0.6745
PR AUC (Average Precision): 0.2186

--- Threshold = 0.50 ---
Confusion Matrix:
 [[12399  5684]
 [  989  1282]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.69      0.79     18083
           1       0.18      0.56      0.28      2271

    accuracy                           0.67     20354
   macro avg       0.56      0.63      0.53     20354
weighted avg       0.84      0.67      0.73     20354


--- Threshold = Top 10% Risk ---
Top 10% threshold value: 0.6441
Confusion Matrix:
 [[16595  1488]
 [ 1723   548]]

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.92      0.91     18083
           1       0.27      0.24      0.25      2271

    accuracy                           0.84     20354
   macro avg       0.59      0.58      0.58     20354
weighted avg       0.83      0.84      0.84     20354


Readmission rate in top 10% highest-risk group: 0.26